# 1. Importando bibliotecas

In [0]:
import requests 
import pandas as pd
from datetime import datetime

# 2. Extração e Transformação dos dados

In [0]:
def extrair_dados_bitcoin():
    url = 'https://api.coinbase.com/v2/prices/spot' #API da Coinbase para obter o preço atual do Bitcoin
    response = requests.get(url)
    return response.json()

In [0]:
dados_bitcoin = extrair_dados_bitcoin()
print(dados_bitcoin)

In [0]:
def extrair_cotacao_usd_brl():
    api_key = '2291516f22814424a5eca3af2106d9e0'
    url = f'https://api.currencyfreaks.com/v2.0/rates/latest?apikey={api_key}'
    response = requests.get(url)
    return response.json()

In [0]:
dados_cotacao = extrair_cotacao_usd_brl()
print(dados_cotacao['rates']['BRL'])

In [0]:
def tratar_dados_bitcoin(dados_json, taxa_usd_brl):
    valor_usd = float(dados_json['data']['amount'])
    criptomoeda = dados_json['data']['base']
    moeda_original = dados_json['data']['currency']

    #convertendo de USD para BRL
    valor_brl = valor_usd * taxa_usd_brl

    #adicoinando timesptamp
    timestamp = datetime.now()

    dado_tratado = [{
        "valor_usd": valor_usd,
        "valor_brl": valor_brl,
        "criptomoeda": criptomoeda,
        "moeda_original": moeda_original,
        "taxa_conversao_usd_brl": taxa_usd_brl,
        "timestamp": timestamp
    }]

    return dado_tratado

In [0]:
#extraindo os dados
dados_bitcoin = extrair_dados_bitcoin()
dados_cotacao = extrair_cotacao_usd_brl()

#extraindo a taxa de conversão de UDS para BRL
taxa_usd_brl = float(dados_cotacao['rates']['BRL'])

#tratando os dados e convertendo para BRL
dados_bitcoin_tratado = tratar_dados_bitcoin(dados_bitcoin, taxa_usd_brl)

print(dados_bitcoin_tratado)

# 3. Configurando Unity Catalog

In [0]:
%sql
create catalog if not exists pipe_api_bitcoin
comment "Catálogo de demonstração do workshop Pipeline de dados API Bitcoin"

In [0]:
%sql
create schema if not exists pipe_api_bitcoin.lakehouse
comment "Schema Lakehouse para salvar dados processados";

In [0]:
%sql
create volume if not exists pipe_api_bitcoin.lakehouse.raw_files
comment "Volume para arquivos brutos de ingestão inicial";

In [0]:
%sql
create schema if not exists pipe_api_bitcoin.bitcoin_data
comment "Schema para armazenar dados de Bitcoin processados";

# 4. Criando Pandas DF

In [0]:
#Criando DF Pandas
df_pandas = pd.DataFrame(dados_bitcoin_tratado)

#validação com view dos dados
print(df_pandas)

# 05. Salvando em Json usando Pandas

In [0]:
# Pegando o timestamp do próprio evento
events_ts = dados_bitcoin_tratado[0]["timestamp"]

#Convertendo o nome para formato seguro no arquivo
ts = events_ts.strftime("%Y%m%d_%H%M%S_%F")

# Caminho do arquivo JSON
json_file = f"/Volumes/pipe_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.json"

#Salvando JSON usando com pandas
df_pandas.to_json(json_file, orient='records', date_format='iso', indent=2)
print(f"✅Arquivo JSON salvo em {json_file}")


#06. Salvando CSV usando PySpark

In [0]:
#Caminho do arquivo CSV
csv_file = f"/Volumes/pipe_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.csv"

#Salvando CSV usando com pandas
df_pandas.to_csv(csv_file, index=False)
print(f"✅Arquivo CSV salvo em {csv_file}")

# 07. Salvando como Parquet usando PySpark

In [0]:
#Caminho do arquivo em Parquet
parquet_file = f"/Volumes/pipe_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.parquet"

#Salvando como Parquet usando com pandas
df_pandas.to_parquet(parquet_file, index=False)
print(f"✅Arquivo Parquet salvo: {parquet_file}")

# 8. Salvando como Delta Table

In [0]:
#Convertendo o DF Pandas para PySpark
df_spark = spark.createDataFrame(df_pandas)

#Escrevendo no Lakehouse
df_spark.printSchema()

# 9. Convertendo Delta Table para DF

In [0]:
df_spark.write.format('delta').mode('append').saveAsTable('pipe_api_bitcoin.bitcoin_data.bitcoin_data')


### Consultando Delta Table com SQL

In [0]:
%sql
select * from pipe_api_bitcoin.bitcoin_data.bitcoin_data

### Verificando histórico da Delta Table (time travel)

In [0]:
%sql
describe history pipe_api_bitcoin.bitcoin_data.bitcoin_data

#10. Resumo do Pipeline
Este pipeline completo realiza:
1. **Extração**: Busca dados da API Coinbase e cotação USD-BRL
2. ✅ **Transformação**: Trata dados, converte para BRL e adiciona timestamp
3. ✅ **Infraestrutura**: Cria Catalog e Schema Lakehouse no Unity Catalog
4. ✅ **Carga em múltiplos formatos usando PySpark**:
    - **JSON**: Formato texto legível, ideal para dados brutos
    - **CSV**: Formato texto universal, legível por humanos (salvo com PySpark)
    - **Parquet**: Formato binário colunar, otimizado para Big Data (salvo com PySpark)
    - **Delta Table**: Formato com ACID transactions e time travel
5. ✅ **Conversão**: Delta Table → Spark DataFrame

**Comparação de Formatos:**
- 📄 **CSV**: Legível, maior, mais lento → Use para debugging e dados pequenos
- 📊 **Parquet**: Binário, menor, mais rápido → Use para Big Data e analytics
- 🎯 **Delta Table**: Parquet + ACID + Time Travel → Use para Data Warehouses e pipelines críticos

**Próximos passos**: Criar dashboard e agente de IA para análise dos dados!